# 10 — Train DINOv2-S baseline (fold 0 smoke)

Session options: **GPU T4** (x1 enough; x2 OK), **Internet ON** (needed once for torch.hub model code).

In [ ]:
from pathlib import Path
import sys

def find_dir_with(*parts: str) -> Path | None:
    needle = Path(*parts)
    for p in Path('/kaggle/input').rglob(parts[-1]):
        if p.is_dir() and p.name == parts[-1]:
            # verify parent chain loosely
            return p
    return None

# Code dataset
REPO = Path('/kaggle/input/datasets/girishbose/rsna-knee-code')
if not (REPO / 'src' / 'rsna_knee').exists():
    hits = [h for h in Path('/kaggle/input').rglob('rsna_knee') if h.is_dir() and h.parent.name == 'src']
    REPO = hits[0].parent.parent

# Cache: notebook output often lands under /kaggle/input/notebooks/.../cache_v1 or flat *.npz
CACHE = None
for cand in [
    Path('/kaggle/input/notebooks/girishbose/rsna-knee-cache-v1/cache_v1'),
    Path('/kaggle/input/notebooks/girishbose/rsna-knee-cache-v1'),
    Path('/kaggle/input/datasets/girishbose/rsna-knee-cache-v1/cache_v1'),
    Path('/kaggle/input/datasets/girishbose/rsna-knee-cache-v1'),
]:
    if cand.exists() and (any(cand.glob('*.npz')) or any(cand.rglob('*.npz'))):
        CACHE = cand if any(cand.glob('*.npz')) else next(p for p in cand.rglob('*.npz')).parent
        break
if CACHE is None:
    npzs = list(Path('/kaggle/input').rglob('*.npz'))
    if not npzs:
        raise SystemExit('No cache npz found — check rsna-knee-cache-v1 is attached')
    CACHE = npzs[0].parent

DATA = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')
if not (DATA / 'train.csv').exists():
    DATA = next(p.parent for p in Path('/kaggle/input').rglob('train.csv') if 'rsna-knee' in str(p))

WEIGHTS = next(Path('/kaggle/input').rglob('dinov2_vits14_pretrain.pth'), None)
FOLDS = REPO / 'data' / 'folds' / 'folds_v1.csv'
WEAK = REPO / 'data' / 'processed' / 'weak_labels_v1.csv'
OUT = Path('/kaggle/working/baseline_dinov2_s')
OUT.mkdir(exist_ok=True)

sys.path.insert(0, str(REPO / 'src'))
print('REPO', REPO)
print('CACHE', CACHE, 'n_npz', len(list(CACHE.glob('*.npz'))))
print('DATA', DATA)
print('WEIGHTS', WEIGHTS)
print('FOLDS', FOLDS.exists(), 'WEAK', WEAK.exists())

In [ ]:
import os, subprocess

%pip -q install pyyaml scikit-learn tqdm

env = os.environ.copy()
env['PYTHONPATH'] = str(REPO / 'src') + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')

FOLD = 0
EPOCHS = 3  # smoke; later use 5

cmd = [
    sys.executable, str(REPO / 'scripts' / 'train_baseline_fold.py'),
    '--config', str(REPO / 'configs' / 'baseline_dinov2_s.yaml'),
    '--train-csv', str(DATA / 'train.csv'),
    '--folds', str(FOLDS),
    '--cache-dir', str(CACHE),
    '--fold', str(FOLD),
    '--epochs', str(EPOCHS),
    '--out-dir', str(OUT),
    '--device', 'cuda',
]
if WEIGHTS:
    cmd += ['--weights', str(WEIGHTS)]
if WEAK.exists():
    cmd += ['--weak-csv', str(WEAK)]

print(' '.join(cmd))
subprocess.check_call(cmd, env=env)
print('outputs', list(OUT.iterdir()))